In [1]:
# Cell 1 — imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [2]:
# Cell 2 — load predictions and segment by risk
cust = pd.read_csv('../data/processed/customer_churn_lgbm_predictions.csv')
emp  = pd.read_csv('../data/processed/employee_churn_lgbm_predictions.csv')

# Assign risk tiers
def assign_risk(prob):
    if prob >= 0.7: return 'High risk'
    elif prob >= 0.4: return 'Medium risk'
    else: return 'Low risk'

cust['risk_tier'] = cust['churn_probability'].apply(assign_risk)
emp['risk_tier']  = emp['churn_probability'].apply(assign_risk)

print("Customer risk distribution:")
print(cust['risk_tier'].value_counts())
print("\nEmployee risk distribution:")
print(emp['risk_tier'].value_counts())

Customer risk distribution:
risk_tier
Low risk     1035
High risk     374
Name: count, dtype: int64

Employee risk distribution:
risk_tier
Low risk     247
High risk     47
Name: count, dtype: int64


In [3]:
# Cell 3 — retention action plan table
retention_plan = {
    'Customer — High risk': {
        'trigger': 'Churn probability >= 70%',
        'action': 'Immediate outreach — offer contract upgrade with 20% discount',
        'owner': 'Customer Success team',
        'timeline': 'Within 24 hours of flag'
    },
    'Customer — Medium risk': {
        'trigger': 'Churn probability 40–70%',
        'action': 'Automated email — personalised plan upgrade offer',
        'owner': 'Marketing automation',
        'timeline': 'Within 7 days'
    },
    'Employee — High risk': {
        'trigger': 'Attrition probability >= 70%',
        'action': 'Manager 1:1 conversation + compensation review + workload audit',
        'owner': 'Direct manager + HR',
        'timeline': 'Within 48 hours of flag'
    },
    'Employee — Medium risk': {
        'trigger': 'Attrition probability 40–70%',
        'action': 'Career development conversation + overtime review',
        'owner': 'HR business partner',
        'timeline': 'Within 2 weeks'
    }
}
plan_df = pd.DataFrame(retention_plan).T.reset_index()
plan_df.columns = ['risk_group','trigger','action','owner','timeline']
display(plan_df)
plan_df.to_csv('../data/processed/retention_plan.csv', index=False)

,risk_group,trigger,action,owner,timeline
0,Customer — High risk,Churn probability >= 70%,Immediate outreach — offer contract upgrade wi...,Customer Success team,Within 24 hours of flag
1,Customer — Medium risk,Churn probability 40–70%,Automated email — personalised plan upgrade offer,Marketing automation,Within 7 days
2,Employee — High risk,Attrition probability >= 70%,Manager 1:1 conversation + compensation review...,Direct manager + HR,Within 48 hours of flag
3,Employee — Medium risk,Attrition probability 40–70%,Career development conversation + overtime review,HR business partner,Within 2 weeks


In [5]:
# Cell 4 — estimated revenue at risk from high-risk customers
high_risk_customers = cust[cust['risk_tier']=='High risk']
print(f"\nHigh-risk customers: {len(high_risk_customers)}")

# if MonthlyCharges is in test set — estimate annual revenue at risk
if 'MonthlyCharges' in high_risk_customers.columns:
    revenue_at_risk = high_risk_customers['MonthlyCharges'].sum() * 12
    print(f"Annual revenue at risk: ${revenue_at_risk:,.0f}")


High-risk customers: 374
Annual revenue at risk: $326,579


In [6]:
# Cell 5 — final summary print
print("""
╔══════════════════════════════════════════════════════════╗
║       CHURN PREDICTION — EXECUTIVE SUMMARY               ║
╚══════════════════════════════════════════════════════════╝
""")
for _, row in plan_df.iterrows():
    print(f"[{row['risk_group']}]")
    print(f"  Trigger : {row['trigger']}")
    print(f"  Action  : {row['action']}")
    print(f"  Owner   : {row['owner']}")
    print(f"  By when : {row['timeline']}\n")


╔══════════════════════════════════════════════════════════╗
║       CHURN PREDICTION — EXECUTIVE SUMMARY               ║
╚══════════════════════════════════════════════════════════╝

[Customer — High risk]
  Trigger : Churn probability >= 70%
  Action  : Immediate outreach — offer contract upgrade with 20% discount
  Owner   : Customer Success team
  By when : Within 24 hours of flag

[Customer — Medium risk]
  Trigger : Churn probability 40–70%
  Action  : Automated email — personalised plan upgrade offer
  Owner   : Marketing automation
  By when : Within 7 days

[Employee — High risk]
  Trigger : Attrition probability >= 70%
  Action  : Manager 1:1 conversation + compensation review + workload audit
  Owner   : Direct manager + HR
  By when : Within 48 hours of flag

[Employee — Medium risk]
  Trigger : Attrition probability 40–70%
  Action  : Career development conversation + overtime review
  Owner   : HR business partner
  By when : Within 2 weeks

